In [8]:
%pip install pandas numpy scikit-learn matplotlib seaborn jupyter torch mlflow
import pandas as pd
import numpy as np
import pickle, os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import mlflow
import copy

DATA_PATH  = '../data/'
MODEL_PATH = '../model/'
MAX_VOCAB  = 15000
MAX_LEN    = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Imports done! Using device: {device}")


  Using cached mlflow-3.10.1-py3-none-any.whl.metadata (31 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached mlflow_skinny-3.10.1-py3-none-any.whl.metadata (32 kB)
  Using cached mlflow_tracing-3.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.2-py3-none-any.whl.metadata (5.3 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.4-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-46.0.6-cp311-abi3-win_amd64.whl.metadata (5.7 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached huey-2.6.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached skops-0.13.0-py3-none-any.whl.metadata (5.6 kB)
  Using cached waitress-3.0.2-py3-none-any.whl.metadata (5.8 kB)
  Using cac

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

Imports done! Using device: cpu


In [9]:
pairs_df = pd.read_csv(f'{DATA_PATH}pairs.csv')
products = pd.read_csv(f'{DATA_PATH}products.csv')

id_to_name = dict(zip(products['product_id'], products['product_name']))
pairs_df['name_1'] = pairs_df['product_1'].map(id_to_name)
pairs_df['name_2'] = pairs_df['product_2'].map(id_to_name)
pairs_df = pairs_df.dropna()
print(f"Pairs ready: {len(pairs_df)}")
print(pairs_df.head())


Pairs ready: 200000
   product_1  product_2  label                                     name_1  \
0        400      35422      0        Grass-Fed Yogurt Blueberry Cardamom   
1      30119      13225      1    Organic French Style Meyer Lemon Yogurt   
2      32850       2360      0                      Organic Pumpkin Puree   
3      46274      19734      1  Ready Snax Veggie & Cheese With Ranch Dip   
4      30635      42342      1                  Lactose Free Yogurt Plain   

                                              name_2  
0  Gentle And Predictable Overnight Relief Laxati...  
1                                        Lowfat Milk  
2                                    Lorraine Quiche  
3                                Classic Mix Variety  
4                              Roasted Turkey Breast  


In [10]:
import collections

class SimpleTokenizer:
    def __init__(self, num_words=None):
        self.num_words = num_words
        self.word_index = {}
        self.index_word = {}
        self.vocab_size = 1 # reserve 0 for padding
        
    def fit_on_texts(self, texts):
        counter = collections.Counter()
        for text in texts:
            words = str(text).lower().split()
            counter.update(words)
            
        most_common = counter.most_common(self.num_words - 1 if self.num_words else None)
        for idx, (word, _) in enumerate(most_common, start=1):
            self.word_index[word] = idx
            self.index_word[idx] = word
        self.vocab_size = len(self.word_index) + 1
            
    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            words = str(text).lower().split()
            seq = [self.word_index[w] for w in words if w in self.word_index]
            sequences.append(seq)
        return sequences

def pad_sequences(sequences, maxlen):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int32)
    for i, seq in enumerate(sequences):
        length = min(len(seq), maxlen)
        if length > 0:
            padded[i, -length:] = seq[:length] # pre-padding
    return padded

tokenizer = SimpleTokenizer(num_words=MAX_VOCAB)
tokenizer.fit_on_texts(list(pairs_df['name_1']) + list(pairs_df['name_2']))

X_left  = pad_sequences(tokenizer.texts_to_sequences(pairs_df['name_1']), maxlen=MAX_LEN)
X_right = pad_sequences(tokenizer.texts_to_sequences(pairs_df['name_2']), maxlen=MAX_LEN)
y = pairs_df['label'].values

# Save tokenizer
with open(f'{MODEL_PATH}tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print(f"X_left shape:  {X_left.shape}")
print(f"X_right shape: {X_right.shape}")
print("Tokenizer saved!")


X_left shape:  (200000, 12)
X_right shape: (200000, 12)
Tokenizer saved!


In [11]:
X_l_tr, X_l_te, X_r_tr, X_r_te, y_tr, y_te = train_test_split(
    X_left, X_right, y,
    test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_l_tr_t = torch.tensor(X_l_tr, dtype=torch.long)
X_r_tr_t = torch.tensor(X_r_tr, dtype=torch.long)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32).unsqueeze(1)

X_l_te_t = torch.tensor(X_l_te, dtype=torch.long)
X_r_te_t = torch.tensor(X_r_te, dtype=torch.long)
y_te_t = torch.tensor(y_te, dtype=torch.float32).unsqueeze(1)

print(f"Train: {len(y_tr)} | Test: {len(y_te)}")


Train: 160000 | Test: 40000


In [12]:
class SiameseNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, max_len=12):
        super(SiameseNetwork, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.lstm1 = nn.LSTM(embedding_dim, 128, batch_first=True)
        self.dropout1 = nn.Dropout(0.3)
        
        self.lstm2 = nn.LSTM(128, 64, batch_first=True)
        self.batch_norm = nn.BatchNorm1d(64)
        
        self.fc1 = nn.Linear(64, 64)
        self.dropout2 = nn.Dropout(0.2)
        self.out = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward_once(self, x):
        x = self.embedding(x)
        x, _ = self.lstm1(x)
        x = self.dropout1(x)
        _, (hn, _) = self.lstm2(x)
        x = hn[-1] # take last hidden state
        x = self.batch_norm(x)
        return x
        
    def forward(self, x1, x2):
        out1 = self.forward_once(x1)
        out2 = self.forward_once(x2)
        
        # Manhattan distance
        distance = torch.abs(out1 - out2)
        
        x = torch.relu(self.fc1(distance))
        x = self.dropout2(x)
        x = self.sigmoid(self.out(x))
        return x

model = SiameseNetwork(MAX_VOCAB, 128, MAX_LEN).to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)


SiameseNetwork(
  (embedding): Embedding(15000, 128)
  (lstm1): LSTM(128, 128, batch_first=True)
  (dropout1): Dropout(p=0.3, inplace=False)
  (lstm2): LSTM(128, 64, batch_first=True)
  (batch_norm): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (out): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


In [13]:
mlflow.set_experiment("instacart-siamese")

batch_size = 256
epochs = 15
patience = 3

train_dataset = TensorDataset(X_l_tr_t, X_r_tr_t, y_tr_t)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_l_te_t, X_r_te_t, y_te_t)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

with mlflow.start_run():
    mlflow.log_params({
        "epochs": epochs,
        "batch_size": batch_size,
        "lstm_units_1": 128,
        "lstm_units_2": 64,
        "max_vocab": MAX_VOCAB,
        "max_len": MAX_LEN
    })

    best_val_loss = float('inf')
    best_acc = 0.0
    patience_counter = 0
    best_model_weights = copy.deepcopy(model.state_dict())

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        for x1_batch, x2_batch, y_batch in train_loader:
            x1_batch, x2_batch, y_batch = x1_batch.to(device), x2_batch.to(device), y_batch.to(device)
            
            optimizer.zero_grad()
            outputs = model(x1_batch, x2_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x1_batch.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x1_batch, x2_batch, y_batch in test_loader:
                x1_batch, x2_batch, y_batch = x1_batch.to(device), x2_batch.to(device), y_batch.to(device)
                outputs = model(x1_batch, x2_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * x1_batch.size(0)
                
                predicted = (outputs > 0.5).float()
                total += y_batch.size(0)
                correct += (predicted == y_batch).sum().item()
                
        val_loss /= len(test_loader.dataset)
        val_acc = correct / total
        
        print(f"Epoch {epoch+1}/{epochs} - loss: {train_loss:.4f} - val_loss: {val_loss:.4f} - val_accuracy: {val_acc:.4f}")
        
        # Early Stopping & Checkpoint
        if val_acc > best_acc:
            best_acc = val_acc
            best_val_loss = val_loss
            patience_counter = 0
            best_model_weights = copy.deepcopy(model.state_dict())
            torch.save(model.state_dict(), f'{MODEL_PATH}best_model.pt')
            print("  --> Saved best model")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered")
                break
                
    model.load_state_dict(best_model_weights)
    mlflow.log_metric("val_accuracy", best_acc)
    print(f"\nBest validation accuracy: {best_acc:.4f}")

print("Training complete! Model saved to model/best_model.pt")


2026/03/30 02:44:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/30 02:44:26 INFO mlflow.store.db.utils: Updating database tables
2026/03/30 02:44:27 INFO mlflow.tracking.fluent: Experiment with name 'instacart-siamese' does not exist. Creating a new experiment.


Epoch 1/15 - loss: 0.5752 - val_loss: 0.4979 - val_accuracy: 0.7594
  --> Saved best model
Epoch 2/15 - loss: 0.4702 - val_loss: 0.4485 - val_accuracy: 0.7941
  --> Saved best model
Epoch 3/15 - loss: 0.4140 - val_loss: 0.4199 - val_accuracy: 0.8124
  --> Saved best model
Epoch 4/15 - loss: 0.3736 - val_loss: 0.4057 - val_accuracy: 0.8226
  --> Saved best model
Epoch 5/15 - loss: 0.3416 - val_loss: 0.3974 - val_accuracy: 0.8289
  --> Saved best model
Epoch 6/15 - loss: 0.3167 - val_loss: 0.3953 - val_accuracy: 0.8312
  --> Saved best model
Epoch 7/15 - loss: 0.2953 - val_loss: 0.3977 - val_accuracy: 0.8325
  --> Saved best model
Epoch 8/15 - loss: 0.2762 - val_loss: 0.4057 - val_accuracy: 0.8336
  --> Saved best model
Epoch 9/15 - loss: 0.2615 - val_loss: 0.4084 - val_accuracy: 0.8358
  --> Saved best model
Epoch 10/15 - loss: 0.2452 - val_loss: 0.4173 - val_accuracy: 0.8340
Epoch 11/15 - loss: 0.2355 - val_loss: 0.4355 - val_accuracy: 0.8337
Epoch 12/15 - loss: 0.2240 - val_loss: 0.44

In [ ]:
model.eval()
y_pred_list = []
with torch.no_grad():
    for x1_batch, x2_batch, _ in test_loader:
        x1_batch, x2_batch = x1_batch.to(device), x2_batch.
        
        to(device)
        outputs = model(x1_batch, x2_batch)
        preds = (outputs > 0.5).int().cpu().numpy()
        y_pred_list.extend(preds)

y_pred = np.array(y_pred_list).flatten()
print(classification_report(y_te, y_pred))
print(confusion_matrix(y_te, y_pred))


              precision    recall  f1-score   support

           0       0.84      0.82      0.83     19904
           1       0.83      0.85      0.84     20096

    accuracy                           0.84     40000
   macro avg       0.84      0.84      0.84     40000
weighted avg       0.84      0.84      0.84     40000

[[16379  3525]
 [ 3012 17084]]
